## Infrastructure Bootstrap — Why This Pattern Exists

Every production notebook that needs to survive a move from a laptop to a cloud cluster must solve one problem first: *where am I running, and where is my data?*
This bootstrap detects the environment at runtime by probing for `dbutils` — a global that exists only on Databricks — so the same file runs on your WSL2 machine today and on a Databricks cluster tomorrow without a single line change.
Path constants (`DATA_DIR`, `CHECKPOINT_DIR`, `OUTPUT_DIR`) are derived from `BASE_DIR` once and reused everywhere, so there are no hardcoded absolute paths scattered through the notebook.
Finally, `.master("local[*]")` is set *only* when off-Databricks — on a cluster you never override the master, you let the platform wire it for you.

In [1]:
import os
from pyspark.sql import SparkSession

try:
    dbutils  # noqa: F821
    IS_DATABRICKS = True
except NameError:
    IS_DATABRICKS = False

if IS_DATABRICKS:
    BASE_DIR = "/Volumes/workspace/telemetry/telemetryplatform"
    spark = SparkSession.builder.appName("TelemetryPlatform").getOrCreate()
else:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    spark = (
        SparkSession.builder
        .appName("TelemetryPlatform")
        .master("local[*]")
        .getOrCreate()
    )

DATA_DIR       = f"{BASE_DIR}/data"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
OUTPUT_DIR     = f"{BASE_DIR}/output"

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)

print(f"Running on {'Databricks' if IS_DATABRICKS else 'WSL2 local'} | BASE_DIR = {BASE_DIR}")

26/08/07 19:11:53 WARN Utils: Your hostname, G15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/07 19:11:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/07 19:11:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Running on WSL2 local | BASE_DIR = /home/sabya/projects/pyspark-dev


## Table of Contents

| Section | Title |
|---------|-------|
| §4  | Big Data Constraints — Scale-Out Justification via the Three Vs |
| §5  | Consistency Models — ACID vs BASE via CAP Theorem |
| §6  | MapReduce — Total Miles Driven per Vehicle Model |
| §7  | Hadoop vs Spark — In-Memory Computing vs Disk-Bound I/O |
| §8  | Synthetic Dataset Generation |
| §9  | Transformations & Aggregation — Narrow vs Wide Dependencies (Part 1) |
| §10 | Transformations & Aggregation — Narrow vs Wide Dependencies (Part 2) |
| §11 | Transformations & Aggregation — Narrow vs Wide Dependencies (Part 3) |
| §12 | *(coming soon)* |
| §13 | Data Skew — Salting Strategy for Data Skew |
| §14 | RDD Lineage & Fault Tolerance (Part 1) |
| §15 | RDD Lineage & Fault Tolerance (Part 2) |
| §16 | Liability of Lineage & Checkpointing |
| §17 | Lazy Evaluation — Transformations vs Actions |
| §18 | DAG Construction — Wide Dependencies and Physical Execution Stages |
| §19 | Data Locality and the I/O Bandwidth Tax |
| §16b | Liability of Lineage — Checkpointing vs Caching (Part 4, Q3) |
| §20 | *(coming soon)* |
| §21 | *(coming soon)* |


## §4 — Big Data Constraints

> 📌 Assignment Mapping — Part 1, Q1: Scale-Out Justification via the Three Vs

Before you write a single line of Spark, you need to understand *why* Spark exists at all — and the honest answer starts with a physical limit engineers call **the Wall**.

### The Wall: What Vertical Scaling Can't Give You

The instinct when a system gets slow is to buy a bigger machine: more CPU cores, more RAM, faster NVMe disks.
That works, right up until it doesn't.
The first problem is cost — doubling RAM on a server doesn't double its price, it more than doubles it, and the curve gets steeper with every tier.
The second problem is harder: even with unlimited money, a single machine has physical ceilings.
A commodity server today tops out around 6 TB of RAM and a few hundred CPU cores; beyond that you're into exotic hardware with six-figure price tags and specialised support contracts.
The third problem is availability — a single node is a single point of failure, so while you're upgrading it, or when it fails, everything stops.
This is **the Wall**: the point beyond which adding resources to one machine either becomes economically irrational, physically impossible, or both.
For small datasets that Wall is comfortably far away.
For a global vehicle telemetry platform, you hit it almost immediately.

### The Three Vs of This Workload

To see why, apply the **Three Vs** — the standard architectural framework for characterising a big-data problem:

**Volume** is the sheer amount of data at rest.
500,000 vehicles, each emitting GPS coordinates, engine metrics, and battery readings at high frequency, 24 hours a day, 365 days a year.
Even at a conservative 1 KB per event and one event per second per vehicle, that's roughly 43 TB of raw data per year.
No single server reliably holds, indexes, and queries 43 TB while simultaneously ingesting new data at full throughput.

**Velocity** is the rate at which data arrives and must be processed.
GPS and engine telemetry is not a nightly batch dump — it is a continuous, near-real-time stream.
When a vehicle's engine temperature spikes, the platform needs to surface that event in seconds, not hours.
Ingesting 500,000 concurrent streams and running windowed aggregations over them is not a workload any single node can absorb without dropping events or falling behind the stream.

**Variety** is the heterogeneity of the data.
This platform combines structured sensor readings (temperature, speed, battery efficiency), semi-structured GPS traces (latitude/longitude time series), and eventually unstructured event logs from different vehicle firmware versions.
A fixed-schema relational database on one machine would require constant schema migrations and can't easily handle the semi-structured and unstructured parts alongside the structured ones.

### Why Horizontal Scale-Out Is the Only Answer

Once you've named the Three Vs concretely — 43+ TB/year of volume, 500k concurrent streams of velocity, multi-format variety — the architectural conclusion follows directly.
Vertical scaling hits **the Wall** long before the workload does.
Horizontal scale-out — adding commodity nodes to a distributed cluster, partitioning data across them, and running computation where the data lives — is the only approach that scales with the workload rather than against it.
Spark is specifically designed around this model: data is partitioned into RDD/DataFrame splits that each node processes independently, and the cluster can grow from 3 nodes to 300 without changing the application code.
For this telemetry platform, horizontal scale-out is not a preference — it is a technical requirement imposed by the Three Vs of the data itself.

## §5 — Consistency Models

> 📌 Assignment Mapping — Part 1, Q2: ACID vs BASE via CAP Theorem

To choose the right consistency model for this platform, start with the theorem that forces the choice in the first place.

### CAP Theorem: The Forcing Function

In 2000, Eric Brewer formalised what distributed systems engineers had been bumping into for years: the **CAP Theorem**.
It states that a distributed data store can guarantee at most two of the following three properties simultaneously:

- **Consistency** — every read receives the most recent write, or an error
- **Availability** — every request receives a non-error response (though it may not be the latest data)
- **Partition tolerance** — the system continues to operate even when network messages between nodes are dropped or delayed

The critical insight is that *partition tolerance is not optional* in a distributed system.
Networks partition — cables fail, data centres lose connectivity, cloud availability zones go dark.
Any system that spans more than one machine must be able to survive a partition.
That means partition tolerance is a given, not a choice, which collapses the CAP triangle into a binary decision: when a partition occurs, do you sacrifice **Consistency** or **Availability**?

### ACID vs BASE: Two Different Answers to That Question

Traditional relational databases answer it by choosing Consistency over Availability — this is the **ACID** model.
**ACID vs BASE** is not a minor implementation detail; it is a fundamental architectural commitment about what the system promises when things go wrong.

ACID (**A**tomicity, **C**onsistency, **I**solation, **D**urability) guarantees that every transaction either fully commits or fully rolls back, that the database is never visible in a half-written state, and that concurrent transactions don't corrupt each other.
In CAP terms, an ACID system under a network partition will refuse to serve stale reads — it returns an error rather than potentially inconsistent data.
That's the right trade-off for a bank: you would rather get an error message than see the wrong account balance.

BASE (**B**asically **A**vailable, **S**oft state, **E**ventually consistent) answers the partition question the other way: keep serving requests, even if the data returned is slightly stale, and let the system converge to a consistent state once the partition heals.
In CAP terms, a BASE system prioritises Availability over Consistency.

### Why This Workload Lands on BASE

Now apply that framework to 500,000 vehicles streaming GPS coordinates and sensor readings 24 hours a day.

First, consider what "strict consistency" would cost here.
Each write — a GPS coordinate, an engine temperature reading — would need to be synchronously confirmed across all replicas before the next write is accepted.
At 500,000 concurrent streams, that synchronisation overhead would create a backlog that grows faster than the system can drain it.
The system would fall behind the stream, accumulate unbounded lag, and eventually drop events.

Second, consider the cost of a slightly stale read in this domain.
If a dashboard shows a vehicle's location as it was 200 milliseconds ago rather than right now, nothing breaks.
A vehicle that was at latitude 37.774929 a moment ago is almost certainly still near that coordinate.
The business logic — routing, geofencing, anomaly detection — tolerates that level of staleness without any meaningful degradation.

Compare that to a bank transfer: a 200-millisecond stale read on an account balance could mean approving a transaction that overdrafts the account.
The domain stakes are completely different.

The conclusion follows from the CAP Theorem directly: because this workload requires partition tolerance (distributed, multi-node cluster), and because **availability** — keeping the ingestion pipeline running and never dropping a telemetry event — is more important than guaranteeing every read reflects the absolute latest write, **BASE is the correct consistency model** for high-velocity GPS and sensor ingestion.
ACID would protect a consistency property this domain does not need, at a throughput cost this domain cannot afford.

## §6 — MapReduce

> 📌 Assignment Mapping — Part 2, Q1: MapReduce Flow for Total Miles per Vehicle Model

MapReduce is the computational model that made distributed batch processing tractable, and before you can understand what Spark improved on, you need to understand how MapReduce actually works — not in the abstract, but applied to the specific aggregation this assignment asks for: **total miles driven per vehicle model**.

The job is this: given the raw telemetry CSV, sum `distance_km` grouped by `vehicle_model` across every row in the dataset.
Walk through each phase of the MapReduce pipeline to see how the framework turns this into a distributed computation.

---

### Phase 1 — Split

The input file (or set of files) is divided into fixed-size **splits** — typically 128 MB chunks in HDFS.
Each split is assigned to one Map task on the cluster node that holds that data block locally, so computation moves to the data rather than the other way around.
For our telemetry CSV this means the ~1M rows are carved into several independent chunks, with no single node needing to read the whole file.

```
telemetry_raw.csv
├── Split 0  → rows 0 – 249,999       (assigned to Node A)
├── Split 1  → rows 250,000 – 499,999 (assigned to Node B)
├── Split 2  → rows 500,000 – 749,999 (assigned to Node C)
└── Split 3  → rows 750,000 – 996,549 (assigned to Node D)
```

---

### Phase 2 — Map

Each Map task reads its split row by row and emits **intermediate key-value pairs**.
For the *total miles driven per vehicle model* aggregation, the mapper's only job is to extract the two relevant fields and discard everything else:

```
Input row:  vehicle_id=VH_0001, vehicle_model="Tesla Model Y", distance_km=312.45, ...
Emits:      ("Tesla Model Y", 312.45)

Input row:  vehicle_id=VH_0243, vehicle_model="Rivian R1T",    distance_km=87.10,  ...
Emits:      ("Rivian R1T", 87.10)

Input row:  vehicle_id=VH_0007, vehicle_model="Tesla Model Y", distance_km=204.88, ...
Emits:      ("Tesla Model Y", 204.88)
```

Every mapper on every node produces a stream of `(vehicle_model, distance_km)` pairs — one pair per input row, nothing more.
The Map phase is embarrassingly parallel: each split is processed independently, with no communication between nodes.

---

### Phase 3 — Shuffle and Sort

This is the phase that costs the most — and it is the phase Spark was specifically designed to minimise.

After all mappers finish, the framework **shuffles** their output across the network so that every pair with the same key lands on the same Reducer node.
All `("Tesla Model Y", ...)` pairs end up on one reducer; all `("Rivian R1T", ...)` pairs on another.
As the pairs arrive, they are **sorted** by key so the reducer receives them as a contiguous, ordered group:

```
Reducer 0 receives (sorted):
    ("BMW iX",               142.30)
    ("BMW iX",               89.77)
    ("BMW iX",               ...)   ← all BMW iX pairs from all splits

Reducer 1 receives (sorted):
    ("Ford Mustang Mach-E",  201.55)
    ("Ford Mustang Mach-E",  ...)   ← all Ford pairs from all splits

Reducer 2 receives (sorted):
    ("Tesla Model Y",        312.45)
    ("Tesla Model Y",        204.88)
    ("Tesla Model Y",        ...)   ← all Tesla Model Y pairs from all splits
    ...
```

The shuffle is a full network data transfer — every mapper writes its output to local disk, and every reducer reads its share over the network.
At scale this is expensive both in time (network latency × data volume) and in I/O (mapper output written to disk before it can be read).

---

### Phase 4 — Reduce

Each Reducer iterates over the sorted pairs for its assigned key and applies the aggregation function — in this case, a simple running sum:

```
Reducer 2 processes "Tesla Model Y":
    running_sum = 0
    + 312.45  →  312.45
    + 204.88  →  517.33
    + ...     →  ...
    Final emit: ("Tesla Model Y", <total_km>)
```

Each reducer emits one final key-value pair per key: `(vehicle_model, total_distance_km)`.
The union of all reducers' output is the answer to the **total miles driven per vehicle model** query — seven rows, one per model in our catalogue.

---

### The Full Pipeline at a Glance

```
CSV splits
    │
    ▼  MAP (per split, parallel)
    ("Tesla Model Y", 312.45)
    ("Rivian R1T",     87.10)
    ("Tesla Model Y", 204.88)
    ...
    │
    ▼  SHUFFLE + SORT (network transfer, disk I/O)
    All ("BMW iX", *)             → Reducer 0
    All ("Ford Mustang Mach-E",*) → Reducer 1
    All ("Tesla Model Y", *)      → Reducer 2
    ...
    │
    ▼  REDUCE (sum per key)
    ("BMW iX",             245,813.44)
    ("Ford Mustang Mach-E",198,204.71)
    ("Tesla Model Y",      312,091.55)
    ...
```

The elegance of MapReduce is that none of the four phases require any node to hold the full dataset in memory — each node processes only its split, emits only what the reducer needs, and the framework handles all coordination.
The cost is the mandatory disk write between Map output and Reduce input, which becomes the bottleneck the moment you need to run this pipeline more than once — as iterative ML algorithms do.

## §7 — Hadoop vs Spark

> 📌 Assignment Mapping — Part 2, Q2: In-Memory Computing vs Disk-Bound I/O

MapReduce is a sound model for one-shot batch aggregations — the kind we just walked through in §6.
But the telemetry platform's roadmap includes predictive-maintenance ML: models that watch engine temperature trends, battery degradation curves, and speed profiles to predict component failures before they happen.
Those algorithms are iterative by nature — gradient descent, k-means, random forest training all make dozens or hundreds of passes over the same dataset.
That is exactly where Hadoop MapReduce breaks down and Spark earns its place.

### Hadoop's Disk-Bound I/O Bottleneck

In the classic Hadoop MapReduce model, every stage in a job writes its output to HDFS before the next stage reads it.
For a single-pass aggregation that cost is acceptable — you pay it once and you're done.
For an iterative ML algorithm with, say, 100 gradient descent iterations, the flow looks like this:

```
Iteration 1:  read from HDFS → compute gradient → write result to HDFS
Iteration 2:  read from HDFS → compute gradient → write result to HDFS
Iteration 3:  read from HDFS → compute gradient → write result to HDFS
...           (×100)
```

Every iteration incurs a full HDFS read and a full HDFS write — replicated across three nodes for fault tolerance.
For a 43 TB telemetry dataset that is 43 TB × 3 replicas read + 43 TB × 3 replicas written, per iteration, for 100 iterations.
The **disk-bound I/O bottleneck** is not a configuration problem you can tune away; it is structural.
The framework was designed around the assumption that data is too large to fit in memory, so disk is the only place to put intermediate results.
That assumption made sense in 2004 when commodity RAM was measured in gigabytes.
It is a serious liability now.

### Spark's In-Memory Computing Model

Spark was designed from the ground up to break that assumption.
Its core abstraction — the Resilient Distributed Dataset (RDD), and its higher-level successor the DataFrame — is an immutable, lazily-evaluated distributed collection that can be **persisted in cluster memory** between operations.

For the same iterative ML workload, the flow becomes:

```
Load dataset → cache in cluster memory  (one disk read)
Iteration 1:  read from memory → compute gradient → keep result in memory
Iteration 2:  read from memory → compute gradient → keep result in memory
Iteration 3:  read from memory → compute gradient → keep result in memory
...           (×100, zero additional disk reads)
```

This is **in-memory computing**: the dataset lives in distributed RAM across the cluster nodes, and each iteration reads from that in-memory representation rather than re-fetching from disk.
The practical speed difference is not incremental — benchmarks consistently show Spark running iterative algorithms 10–100× faster than Hadoop MapReduce on the same hardware, purely because of eliminated disk round-trips.

### Why This Matters Specifically for Predictive Maintenance

The telemetry platform's predictive-maintenance models will be trained on the same columns we're already collecting: `engine_temperature`, `battery_efficiency`, `speed`, `distance_km`, and derived features like rolling averages and rate-of-change metrics.
Training a single model variant — one set of hyperparameters, one algorithm — might require 50–200 iterations over the full dataset.
A production ML pipeline runs dozens of candidate models in parallel (hyperparameter search), retrains on a rolling basis as new fleet data arrives, and evaluates on a held-out validation set after each run.

Under Hadoop's model, each of those training runs would re-read the entire dataset from disk on every iteration.
The engineering team would spend most of their time waiting for disk I/O rather than iterating on the model.
Under Spark's **in-memory computing** model, the training data is loaded once, cached across the cluster, and all iterations — across all candidate models — read from memory.
The feedback loop compresses from hours to minutes.

Hadoop MapReduce is not wrong — it is well-suited for the single-pass ETL and aggregation jobs that dominated early big-data workloads.
But for a platform whose analytical roadmap is anchored in iterative ML, the **disk-bound I/O bottleneck** of MapReduce is a structural mismatch, and Spark's **in-memory computing** model is the correct foundation to build on.

## §8 — Synthetic Dataset Generation

The assignment references a real 500,000-vehicle global fleet.
At full volume that dataset is impractical for local development — Spark startup overhead alone would make every iteration loop painfully slow, and the point of working locally is fast feedback.
Instead we generate a ~1M-row synthetic proxy: large enough for every aggregation, window, and join pattern in the assignment to behave as it would at scale, small enough to run comfortably in a single WSL2 session with `local[*]`.
The seed (`np.random.seed(42)`) makes every column — including the deliberate skew — byte-for-byte reproducible across machines and runs, so the results you see here will match exactly when this notebook is promoted to Databricks.

**On the deliberate skew:** 10 of the 500 vehicle IDs are designated "hot" vehicles and each accumulates **1,000× the record volume** of a typical vehicle.
This is not an accident or a data-quality problem — it is the exact skew scenario that Part 3, Q2 of the assignment asks you to diagnose and fix.
Baking it in here, at generation time, means every subsequent section operates on a dataset that already contains the pathology we will later optimize.

In [2]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

# ── Vehicle catalogue ────────────────────────────────────────────────────────
VEHICLE_MODELS = [
    "Tesla Model 3",
    "Tesla Model Y",
    "Ford Mustang Mach-E",
    "Rivian R1T",
    "Lucid Air",
    "BMW iX",
    "Hyundai IONIQ 6",
]

N_VEHICLES      = 500          # total distinct vehicle IDs
N_HOT_VEHICLES  = 10           # vehicles that generate 1000x the normal volume
ROWS_PER_NORMAL = 95           # rows for each regular vehicle
SKEW_MULTIPLIER = 1000         # exact skew ratio required by Part 3 Q2
ROWS_PER_HOT    = ROWS_PER_NORMAL * SKEW_MULTIPLIER   # 95,000 per hot vehicle

# Assign models deterministically so groupBy("vehicle_model") is stable
all_vehicle_ids = [f"VH_{i:04d}" for i in range(N_VEHICLES)]
hot_vehicle_ids = set(all_vehicle_ids[:N_HOT_VEHICLES])   # VH_0000 … VH_0009

vehicle_model_map = {
    vid: VEHICLE_MODELS[i % len(VEHICLE_MODELS)]
    for i, vid in enumerate(all_vehicle_ids)
}

# ── Row generation ───────────────────────────────────────────────────────────
def make_rows(vehicle_id: str, n_rows: int) -> pd.DataFrame:
    """Generate n_rows of telemetry for a single vehicle."""
    model = vehicle_model_map[vehicle_id]
    base_ts = pd.Timestamp("2024-01-01")
    timestamps = [
        base_ts + pd.Timedelta(seconds=int(s))
        for s in np.random.randint(0, 365 * 24 * 3600, size=n_rows)
    ]
    return pd.DataFrame({
        "vehicle_id"          : vehicle_id,
        "vehicle_model"       : model,
        "timestamp"           : timestamps,
        "engine_temperature"  : np.round(np.random.uniform(60.0, 120.0, n_rows), 2),
        "speed"               : np.round(np.random.uniform(0.0, 200.0, n_rows), 2),
        "latitude"            : np.round(np.random.uniform(-90.0, 90.0, n_rows), 6),
        "longitude"           : np.round(np.random.uniform(-180.0, 180.0, n_rows), 6),
        "battery_efficiency"  : np.round(np.random.uniform(0.60, 1.00, n_rows), 4),
        "distance_km"         : np.round(np.random.uniform(0.0, 500.0, n_rows), 3),
    })

chunks = []
for vid in all_vehicle_ids:
    n = ROWS_PER_HOT if vid in hot_vehicle_ids else ROWS_PER_NORMAL
    chunks.append(make_rows(vid, n))

df_pandas = pd.concat(chunks, ignore_index=True)

# ── Persist to DATA_DIR ──────────────────────────────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
csv_path = f"{DATA_DIR}/telemetry_raw.csv"
df_pandas.to_csv(csv_path, index=False)

# ── Sanity check ─────────────────────────────────────────────────────────────
hot_count    = df_pandas[df_pandas["vehicle_id"].isin(hot_vehicle_ids)].shape[0]
normal_count = df_pandas[~df_pandas["vehicle_id"].isin(hot_vehicle_ids)].shape[0]
actual_ratio = hot_count / N_HOT_VEHICLES / (normal_count / (N_VEHICLES - N_HOT_VEHICLES))

print(f"Total rows          : {len(df_pandas):,}")
print(f"Hot vehicle rows    : {hot_count:,}  ({N_HOT_VEHICLES} vehicles × {ROWS_PER_HOT:,} rows)")
print(f"Normal vehicle rows : {normal_count:,}  ({N_VEHICLES - N_HOT_VEHICLES} vehicles × {ROWS_PER_NORMAL} rows)")
print(f"Verified skew ratio : {actual_ratio:.1f}x  (target = {SKEW_MULTIPLIER}x)")
print(f"Saved to            : {csv_path}")

Total rows          : 996,550
Hot vehicle rows    : 950,000  (10 vehicles × 95,000 rows)
Normal vehicle rows : 46,550  (490 vehicles × 95 rows)
Verified skew ratio : 1000.0x  (target = 1000x)
Saved to            : /home/sabya/projects/pyspark-dev/data/telemetry_raw.csv


In [3]:
# ── Load CSV into Spark ───────────────────────────────────────────────────────
#
# We prove the skew exists NOW — before any optimization work — so we have a
# concrete "before" baseline to reference in Part 3 Q2.  When you later show
# that salting or AQE eliminates the long-tail partition, the improvement is
# measured against this table, not asserted from memory.

from pyspark.sql import functions as F

sdf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/telemetry_raw.csv")
)

print("=== Schema ===")
sdf.printSchema()

print(f"=== Row count: {sdf.count():,} ===")

print("\n=== Top-15 vehicle_ids by record count (skew is visible here) ===")
(
    sdf.groupBy("vehicle_id")
    .count()
    .orderBy(F.col("count").desc())
    .limit(15)
    .show(truncate=False)
)

=== Schema ===
root
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_model: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- engine_temperature: double (nullable = true)
 |-- speed: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- battery_efficiency: double (nullable = true)
 |-- distance_km: double (nullable = true)



=== Row count: 996,550 ===

=== Top-15 vehicle_ids by record count (skew is visible here) ===


+----------+-----+
|vehicle_id|count|
+----------+-----+
|VH_0000   |95000|
|VH_0001   |95000|
|VH_0002   |95000|
|VH_0003   |95000|
|VH_0004   |95000|
|VH_0005   |95000|
|VH_0006   |95000|
|VH_0007   |95000|
|VH_0008   |95000|
|VH_0009   |95000|
|VH_0043   |95   |
|VH_0051   |95   |
|VH_0065   |95   |
|VH_0052   |95   |
|VH_0021   |95   |
+----------+-----+



## §9–11 — Transformations & Aggregation

> 📌 Assignment Mapping — Part 3, Q1: Narrow vs Wide Dependencies

Every transformation you apply to a Spark DataFrame falls into one of two dependency classes, and knowing which class a transformation belongs to is the single most important thing you can learn for reasoning about Spark performance.

**Narrow dependencies** are transformations where each output partition depends on exactly one input partition — no data needs to cross the network.
 `filter`, `select`, `withColumn`, `map`, `flatMap`, and `drop` are all narrow. Spark can pipeline them together within a single stage, processing them in one pass without any inter-node communication.

**Wide dependencies** are transformations where a single output partition may depend on *multiple* input partitions — data from different nodes has to be shuffled across the network and co-located before the operation can complete.
 `groupBy`, `join`, `distinct`, `repartition`, and `orderBy` are all wide. Each wide transformation forces a stage boundary: Spark must materialise the shuffle output before the next stage can begin.

The worked example below computes **average engine temperature per vehicle model**. Every transformation is labelled so the dependency type is never ambiguous.

In summary: **narrow dependencies** pipeline within a stage at zero network cost, while **wide dependencies** cross the network and define stage boundaries.


In [4]:
from pyspark.sql import functions as F

# Re-use the DataFrame already loaded in §8; reload if kernel was restarted.
sdf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/telemetry_raw.csv")
)

# ── NARROW: select ──────────────────────────────────────────────────────────
# Keeps only the two columns we need.  Each partition is transformed in place;
# no data moves between nodes.  This is a narrow dependency.
df_selected = sdf.select("vehicle_model", "engine_temperature")  # narrow dependency

# ── NARROW: filter ──────────────────────────────────────────────────────────
# Discards rows where engine_temperature is outside a plausible operating range.
# Applied per-partition with no cross-node communication.  narrow dependency.
df_filtered = df_selected.filter(                                  # narrow dependency
    (F.col("engine_temperature") >= 60.0) &
    (F.col("engine_temperature") <= 120.0)
)

# ── NARROW: withColumn ──────────────────────────────────────────────────────
# Converts Celsius to Fahrenheit, deriving a new column row-by-row in each
# partition independently.  narrow dependency.
df_with_f = df_filtered.withColumn(                                # narrow dependency
    "engine_temp_f",
    F.round(F.col("engine_temperature") * 9 / 5 + 32, 2)
)

print("Narrow transforms applied: select → filter → withColumn")
print(f"Partition count after narrow transforms: {df_with_f.rdd.getNumPartitions()}")
df_with_f.printSchema()


Narrow transforms applied: select → filter → withColumn
Partition count after narrow transforms: 12
root
 |-- vehicle_model: string (nullable = true)
 |-- engine_temperature: double (nullable = true)
 |-- engine_temp_f: double (nullable = true)



## §10 — Wide Dependency: groupBy + Aggregation

Now comes the wide transformation.
`groupBy("vehicle_model")` must gather all rows that share the same model key onto the same partition — rows that currently live on different nodes across the cluster.
 Spark triggers a **shuffle**: every node writes its local `vehicle_model` buckets to disk, then each reducer node reads its assigned buckets over the network.
 This is a wide dependency because one output partition depends on *many* input partitions.

The `agg(avg(...))` that follows the `groupBy` rides in the same wide stage — it operates on the already-shuffled data, so it adds no additional shuffle cost.
 The final `orderBy` is a second wide transformation: sorting requires all nodes to agree on a global order, which means another shuffle to a single reducer.


In [5]:
# ── WIDE: groupBy + agg ─────────────────────────────────────────────────────
# groupBy shuffles all rows with the same vehicle_model key onto the same
# partition.  Data crosses the network here.  This is a wide dependency.
df_avg_temp = (
    df_with_f
    .groupBy("vehicle_model")                                      # wide dependency
    .agg(
        F.round(F.avg("engine_temperature"), 2).alias("avg_engine_temp_c"),
        F.round(F.avg("engine_temp_f"), 2).alias("avg_engine_temp_f"),
        F.count("*").alias("record_count"),
    )
)

# ── WIDE: orderBy ────────────────────────────────────────────────────────────
# Sorting requires a global order across all partitions — another shuffle.
# This is a wide dependency.
df_avg_temp_sorted = df_avg_temp.orderBy(                          # wide dependency
    F.col("avg_engine_temp_c").desc()
)

print("Average engine temperature per vehicle model (°C and °F):")
df_avg_temp_sorted.show(truncate=False)


Average engine temperature per vehicle model (°C and °F):


+-------------------+-----------------+-----------------+------------+
|vehicle_model      |avg_engine_temp_c|avg_engine_temp_f|record_count|
+-------------------+-----------------+-----------------+------------+
|Tesla Model Y      |90.03            |194.05           |196650      |
|Ford Mustang Mach-E|90.03            |194.05           |196650      |
|BMW iX             |90.03            |194.05           |101650      |
|Tesla Model 3      |90.02            |194.03           |196650      |
|Hyundai IONIQ 6    |90.0             |193.99           |101650      |
|Rivian R1T         |89.97            |193.94           |101650      |
|Lucid Air          |89.92            |193.86           |101650      |
+-------------------+-----------------+-----------------+------------+



## §11 — Dependency Map Summary

The full transformation chain above has exactly two stage boundaries — the two points where a wide dependency forces Spark to materialise a shuffle before continuing:

```
Stage 1 (pipelined, no shuffle — all narrow dependencies):
  read CSV
  └─ select(vehicle_model, engine_temperature)          ← narrow dependency
  └─ filter(60 ≤ engine_temperature ≤ 120)              ← narrow dependency
  └─ withColumn(engine_temp_f)                          ← narrow dependency
       [SHUFFLE BOUNDARY]
Stage 2 (triggered by groupBy — wide dependency):
  └─ groupBy(vehicle_model).agg(avg(...), count(*))     ← wide dependency
       [SHUFFLE BOUNDARY]
Stage 3 (triggered by orderBy — wide dependency):
  └─ orderBy(avg_engine_temp_c DESC)                    ← wide dependency
```

Narrow dependencies are cheap — Spark pipelines them within a single task and they never touch the network.
 Wide dependencies are expensive — each one spills data to disk, transfers it over the network, and reads it back on the receiving node.
 In production, the design goal is to push as many narrow dependencies as possible before the first wide one (so the shuffle operates on a smaller, already-filtered dataset) and to minimise the total number of wide transformations in the DAG.


In [6]:
# Confirm Spark's physical plan shows two exchange (shuffle) operators,
# corresponding to the two wide dependencies identified above.
print("=== Physical Plan (look for Exchange operators marking shuffle boundaries) ===")
df_avg_temp_sorted.explain(mode="formatted")


=== Physical Plan (look for Exchange operators marking shuffle boundaries) ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [2]: [vehicle_model#93, engine_temperature#95]
Batched: false
Location: InMemoryFileIndex [file:/home/sabya/projects/pyspark-dev/data/telemetry_raw.csv]
PushedFilters: [IsNotNull(engine_temperature), GreaterThanOrEqual(engine_temperature,60.0), LessThanOrEqual(engine_temperature,120.0)]
ReadSchema: struct<vehicle_model:string,engine_temperature:double>

(2) Filter
Input [2]: [vehicle_model#93, engine_temperature#95]
Condition : ((isnotnull(engine_temperature#95) AND (engine_temperature#95 >= 60.0)) AND (engine_temperature#95 <= 120.0))

(3) Project
Output [3]: [vehicle_model#93, engine_temperature#95, round((((engine_temperature#95 

## §13 — Data Skew

> 📌 Assignment Mapping — Part 3, Q2: Salting Strategy for Data Skew

The §8 output already showed the pathology: VH_0000 through VH_0009 each produce **1000x more logs** than any other vehicle_id in the fleet.
 That skew ratio was baked in deliberately — it is the exact scenario this question asks you to diagnose and fix.

### Why Skew Kills a groupBy

When you run `groupBy("vehicle_id")` on a skewed dataset, Spark's hash partitioner assigns all rows for a given key to the same partition.
 The 10 hot vehicles end up dominating 10 partitions, while the other 490 partitions each hold only 95 rows.
 The job's wall-clock time is determined by the slowest task — the one processing 95,000 rows — while every other task finishes almost instantly and sits idle.
 This is the skew tax: you're paying for a cluster-sized job but getting single-task throughput.

### Hash vs Range Partitioning

Before fixing the skew, it's worth being explicit about **Hash vs Range** partitioning, because the choice of partitioner determines how bad the skew actually is:

- **Hash partitioning** (Spark's default for `groupBy`): assigns each row to a partition  based on `hash(key) % numPartitions`.  For a uniform key distribution this works well.  For a skewed key like `vehicle_id`, it concentrates all rows of a hot key onto one  partition deterministically — the worst possible outcome for skew.
- **Range partitioning** (used by `orderBy`/`sortBy`): divides the key space into  contiguous ranges and assigns a range to each partition.  This is the right choice when keys have a natural sort order and you need sorted output.  For `vehicle_id` there is no meaningful ordering to exploit — VH_0000 is not  "closer" to VH_0001 than to VH_0499 in any analytical sense.  Range partitioning would not help here and would add sampling overhead for no gain.

**Hash partitioning is the correct choice** for this `groupBy` — but we need to break the skew by changing what we hash on, not by switching the partitioner.
 That is exactly what salting does.

### The Salting Strategy

Salting appends a random integer suffix to the skewed key before the `groupBy`, spreading what was one partition's worth of data across N partitions. After the first aggregation, the salt is stripped and partial results are combined in a second aggregation on the original key.
 The two-pass approach is the price of breaking the skew — but both passes are far cheaper than the single-pass skewed shuffle was.


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── BEFORE salting: measure hash-partition row counts on vehicle_id ─────────
# repartition(N, key) uses the same hash partitioner as groupBy(key) would.
# glom() collapses each partition into a list; map(len) gives its row count.
sdf_vehicle = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/telemetry_raw.csv")
)

NUM_PARTITIONS = sdf_vehicle.rdd.getNumPartitions()

partition_counts_before = (
    sdf_vehicle
    .repartition(NUM_PARTITIONS, "vehicle_id")
    .rdd.glom().map(len).collect()
)
partition_counts_before_sorted = sorted(partition_counts_before, reverse=True)

print("=== Per-partition row counts BEFORE salting (hash on vehicle_id) ===")
print(f"Num partitions : {len(partition_counts_before)}")
print(f"Max partition  : {max(partition_counts_before):,}  rows")
print(f"Min partition  : {min(partition_counts_before):,}  rows")
print(f"Skew ratio     : {max(partition_counts_before)/max(min(partition_counts_before),1):.1f}x")
print(f"All counts     : {partition_counts_before_sorted}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(partition_counts_before_sorted)), partition_counts_before_sorted, color='#d9534f')
ax.set_title('Per-Partition Row Counts — BEFORE Salting')
ax.set_xlabel('Partition index (sorted desc)')
ax.set_ylabel('Row count')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/skew_before.png", dpi=100)
plt.show()
print(f"Chart saved to {OUTPUT_DIR}/skew_before.png")


=== Per-partition row counts BEFORE salting (hash on vehicle_id) ===
Num partitions : 12
Max partition  : 194,370  rows
Min partition  : 3,325  rows
Skew ratio     : 58.5x
All counts     : [194370, 193990, 193325, 99085, 98895, 98705, 98610, 5130, 4085, 3515, 3515, 3325]
Chart saved to /home/sabya/projects/pyspark-dev/output/skew_before.png


/tmp/ipykernel_13192/2371928151.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

SALT_BUCKETS = 10  # spread each hot key across 10 partitions

# ── Step 1: append a random salt suffix to vehicle_id ───────────────────────
# This turns one hot key (e.g. 'VH_0000') into 10 salted keys
# ('VH_0000_0' … 'VH_0000_9'), distributing its 95,000 rows across
# 10 partitions instead of 1.
df_salted = sdf_vehicle.withColumn(
    "vehicle_id_salted",
    F.concat(
        F.col("vehicle_id"),
        F.lit("_"),
        (F.rand(seed=42) * SALT_BUCKETS).cast("int").cast("string")
    )
)

# ── Step 2: first aggregation on the salted key (wide dependency) ───────────
# Shuffle now operates on salted keys — the hot vehicle's rows are spread
# across 10 partitions, each handling ~9,500 rows instead of 95,000.
df_partial = (
    df_salted
    .groupBy("vehicle_id_salted", "vehicle_id")          # wide dependency
    .agg(
        F.sum("distance_km").alias("partial_distance_km"),
        F.count("*").alias("partial_count"),
    )
)

# ── Step 3: de-salt — strip the suffix and combine partial results ───────────
# Second groupBy on the original vehicle_id; each key now has at most
# SALT_BUCKETS rows coming in, so this shuffle is tiny.
df_final = (
    df_partial
    .groupBy("vehicle_id")                                # wide dependency (de-salt)
    .agg(
        F.sum("partial_distance_km").alias("total_distance_km"),
        F.sum("partial_count").alias("total_count"),
    )
    .orderBy(F.col("total_distance_km").desc())
)

print("=== Top 15 vehicles by total distance (after salting) ===")
df_final.show(15, truncate=False)


=== Top 15 vehicles by total distance (after salting) ===


+----------+--------------------+-----------+
|vehicle_id|total_distance_km   |total_count|
+----------+--------------------+-----------+
|VH_0004   |2.3849069828000013E7|95000      |
|VH_0005   |2.3827319283000007E7|95000      |
|VH_0008   |2.3798489641999997E7|95000      |
|VH_0003   |2.3775278412000008E7|95000      |
|VH_0000   |2.3757081812000006E7|95000      |
|VH_0009   |2.3733391165999975E7|95000      |
|VH_0001   |2.369640304900001E7 |95000      |
|VH_0006   |2.369347926599999E7 |95000      |
|VH_0007   |2.3679173259999998E7|95000      |
|VH_0002   |2.367906317899998E7 |95000      |
|VH_0196   |28301.976           |95         |
|VH_0308   |27184.124           |95         |
|VH_0204   |27150.21            |95         |
|VH_0457   |26842.884000000002  |95         |
|VH_0428   |26797.71            |95         |
+----------+--------------------+-----------+
only showing top 15 rows



In [9]:
# ── AFTER salting: hash-partition on vehicle_id_salted (the fixed shuffle key) ─
partition_counts_after = (
    df_salted
    .repartition(NUM_PARTITIONS, "vehicle_id_salted")
    .rdd.glom().map(len).collect()
)
partition_counts_after_sorted = sorted(partition_counts_after, reverse=True)

print("=== Per-partition row counts AFTER salting (hash on vehicle_id_salted) ===")
print(f"Num partitions : {len(partition_counts_after)}")
print(f"Max partition  : {max(partition_counts_after):,}  rows")
print(f"Min partition  : {min(partition_counts_after):,}  rows")
print(f"Skew ratio     : {max(partition_counts_after)/max(min(partition_counts_after),1):.1f}x")

# ── Side-by-side comparison ──────────────────────────────────────────────────
print("\n=== Before vs After (sorted partition sizes) ===")
print(f"{'Partition':<12} {'Before':>10} {'After':>10}")
for i, (b, a) in enumerate(zip(partition_counts_before_sorted,
                                partition_counts_after_sorted)):
    print(f"{i:<12} {b:>10,} {a:>10,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=False)
axes[0].bar(range(len(partition_counts_before_sorted)),
            partition_counts_before_sorted, color='#d9534f')
axes[0].set_title('BEFORE Salting')
axes[0].set_xlabel('Partition (sorted desc)')
axes[0].set_ylabel('Row count')
axes[1].bar(range(len(partition_counts_after_sorted)),
            partition_counts_after_sorted, color='#5cb85c')
axes[1].set_title('AFTER Salting')
axes[1].set_xlabel('Partition (sorted desc)')
plt.suptitle('Per-Partition Row Counts: Skew Before vs After Salting')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/skew_comparison.png", dpi=100)
plt.show()
print(f"Chart saved to {OUTPUT_DIR}/skew_comparison.png")


=== Per-partition row counts AFTER salting (hash on vehicle_id_salted) ===
Num partitions : 12
Max partition  : 146,822  rows
Min partition  : 41,929  rows
Skew ratio     : 3.5x

=== Before vs After (sorted partition sizes) ===
Partition        Before      After
0               194,370    146,822
1               193,990    146,173
2               193,325     99,086
3                99,085     98,494
4                98,895     79,246
5                98,705     70,789
6                98,610     70,511
7                 5,130     70,309
8                 4,085     61,085
9                 3,515     61,013
10                3,515     51,093
11                3,325     41,929


Chart saved to /home/sabya/projects/pyspark-dev/output/skew_comparison.png


/tmp/ipykernel_13192/3817006626.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## §14–15 — RDD Lineage & Fault Tolerance

> 📌 Assignment Mapping — Part 3, Q3: Fault Tolerance via RDD Lineage

Spark's fault tolerance story is unusual in distributed computing because it does not rely on replicating data to survive node failures. Understanding why requires understanding two design decisions that Spark baked in from the start: immutability and lineage.

### RDDs Are Immutable

Every RDD (and by extension every DataFrame, which is built on top of an RDD) is **immutable** — once created, its contents never change. Transformations don't modify an existing RDD; they produce a new one. `df.filter(...)` returns a new DataFrame that describes the filtered result; the original `df` is unchanged.

Immutability has a critical consequence: Spark never has to worry about a partially-written or corrupted intermediate state. If a computation is interrupted mid-way, the input data is still exactly as it was before the interrupted transformation started.

### Lineage: The DAG as a Recovery Plan

When you chain transformations — `read → filter → withColumn → groupBy → agg` — Spark does not execute them immediately. Instead, it records the sequence as a **lineage graph** (a DAG of transformation steps) and defers execution until an action (like `.collect()` or `.show()`) is called.

That lineage graph is Spark's recovery plan. If a node fails mid-execution and a partition is lost, Spark doesn't need to restore it from a replica — it just re-runs the lineage for that partition on another node, starting from the last stable input.

This is how Spark achieves **fault tolerance** **without heavy data replication**: instead of keeping two or three physical copies of every partition on disk at all times (the HDFS approach), Spark keeps one logical description of how to reconstruct any partition from scratch. Storage cost is O(lineage depth), not O(data size × replication factor).

The trade-off is that reconstruction time grows with lineage depth. For a two-step pipeline, re-running from the source is trivial. For a 200-step iterative pipeline, re-running from scratch after a node failure is expensive — which is exactly the problem that checkpointing (§16) solves.


## §15 — Lineage in Action

The code below builds a short transformation chain and then prints the lineage (`toDebugString`) so you can see the DAG structure as Spark records it. Notice that nothing executes until the final `.count()` action — all the transformations before it are just nodes in the lineage graph.


In [10]:
# Build a deliberate 4-step transformation chain to make the lineage visible.
df_step1 = sdf_vehicle.filter(F.col("speed") > 0)               # narrow
df_step2 = df_step1.withColumn(                                   # narrow
    "speed_ms", F.round(F.col("speed") / 3.6, 4)
)
df_step3 = df_step2.filter(F.col("battery_efficiency") > 0.7)    # narrow
df_step4 = df_step3.groupBy("vehicle_model").agg(                 # wide
    F.avg("speed_ms").alias("avg_speed_ms"),
    F.count("*").alias("n")
)

print("=== RDD Lineage (toDebugString) ===")
print(df_step4.rdd.toDebugString().decode())

# Action triggers execution of the entire lineage.
result_count = df_step4.count()
print(f"\nAction triggered — {result_count} rows produced.")
print("If any partition had failed during this action, Spark would have")
print("re-run only the failed partition's lineage on another node — ")
print("fault tolerance without heavy data replication.")


=== RDD Lineage (toDebugString) ===


(1) MapPartitionsRDD[100] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[99] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  SQLExecutionRDD[98] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[97] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  ShuffledRowRDD[96] at javaToPython at NativeMethodAccessorImpl.java:0 []
 +-(12) MapPartitionsRDD[95] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   MapPartitionsRDD[94] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   MapPartitionsRDD[93] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   FileScanRDD[92] at javaToPython at NativeMethodAccessorImpl.java:0 []



Action triggered — 7 rows produced.
If any partition had failed during this action, Spark would have
re-run only the failed partition's lineage on another node — 
fault tolerance without heavy data replication.


## §16 — Liability of Lineage & Checkpointing

> 📌 Assignment Mapping — Part 3, Q4: Checkpointing to Prevent StackOverflow

Lineage is Spark's recovery superpower — but it has a liability that only shows up in iterative workloads, and it can kill your job in a way that's genuinely surprising the first time you hit it.

Every transformation appends a new node to the lineage DAG. For a short pipeline that DAG is a few nodes deep and costs almost nothing to maintain. For an iterative loop — say, 50 rounds of feature engineering where each round builds on the previous DataFrame — the DAG grows one level deeper per iteration. After 50 iterations the lineage is 50 nodes deep; after 200 it is 200 nodes deep.

Two problems compound as the DAG grows:

1. **StackOverflow errors** — Spark's DAG scheduler recurses through the lineage graph   to plan execution.   A sufficiently deep lineage graph overflows the JVM call stack and crashes the driver   with a `StackOverflow` error — not an out-of-memory error, not a timeout, a stack   overflow.   This is one of the most confusing Spark failures because nothing about the error   message obviously points to lineage depth.
2. **Recovery time grows with depth** — if a node fails late in a 200-iteration run,   Spark must replay the entire 200-step lineage to reconstruct the lost partition.   The longer the lineage, the more work a failure forces.

### Checkpointing Truncates the DAG

`df.checkpoint()` materialises the DataFrame to the checkpoint directory (set in the bootstrap cell as `CHECKPOINT_DIR`) and **severs the lineage**. The checkpointed DataFrame's parent is the checkpoint file on disk, not the chain of transformations that produced it. All history before the checkpoint is forgotten from Spark's perspective.

Called every N iterations, checkpointing keeps the lineage depth bounded at N nodes regardless of how many total iterations run. This prevents **StackOverflow errors** and **stabilizes recovery time**: a failure at any point only requires replaying back to the most recent checkpoint, not back to the beginning of the job.

*(The deeper comparison between `.checkpoint()` and `.cache()/.persist()` — when each is appropriate, and what they cost — is in §16b, Part 4 Q3.)*


In [11]:
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

ITERATIONS   = 40   # enough to build a visibly deep lineage
CKPT_EVERY   = 10   # checkpoint every 10 iterations to bound DAG depth

# Start from a clean, cached baseline so I/O doesn't dominate the timing.
df_iter = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/telemetry_raw.csv")
    .select("vehicle_id", "vehicle_model", "engine_temperature",
            "battery_efficiency", "distance_km")
    .cache()   # cache the source; iterations build on top of it
)
df_iter.count()  # materialise cache

print(f"Simulating {ITERATIONS} iterative feature-engineering passes ...")
print(f"Checkpoint every {CKPT_EVERY} iterations.\n")

for i in range(1, ITERATIONS + 1):
    # Each iteration adds two narrow transformations to the lineage.
    df_iter = df_iter.withColumn(
        "engine_temperature",
        F.round(F.col("engine_temperature") * 0.9999 + 0.001, 4)
    )
    df_iter = df_iter.withColumn(
        "distance_km",
        F.round(F.col("distance_km") + 0.001, 4)
    )

    if i % CKPT_EVERY == 0:
        # Show lineage depth just BEFORE checkpointing.
        debug_str = df_iter.rdd.toDebugString().decode()
        depth = debug_str.count("MapPartitionsRDD")
        print(f"Iteration {i:3d} | lineage depth ~ {depth} nodes  → checkpointing ...")
        df_iter = df_iter.checkpoint()   # truncate the DAG here
        debug_after = df_iter.rdd.toDebugString().decode()
        depth_after = debug_after.count("MapPartitionsRDD")
        print(f"             | lineage depth after checkpoint: {depth_after} nodes")

print("\nFinal lineage (post-checkpoint — should be shallow):")
print(df_iter.rdd.toDebugString().decode())

print("\nSample of final DataFrame values after iterative updates:")
df_iter.show(5, truncate=False)


Simulating 40 iterative feature-engineering passes ...
Checkpoint every 10 iterations.



Iteration  10 | lineage depth ~ 9 nodes  → checkpointing ...


             | lineage depth after checkpoint: 4 nodes
Iteration  20 | lineage depth ~ 4 nodes  → checkpointing ...


             | lineage depth after checkpoint: 4 nodes
Iteration  30 | lineage depth ~ 4 nodes  → checkpointing ...


             | lineage depth after checkpoint: 4 nodes
Iteration  40 | lineage depth ~ 4 nodes  → checkpointing ...


             | lineage depth after checkpoint: 4 nodes

Final lineage (post-checkpoint — should be shallow):
(12) MapPartitionsRDD[179] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |   MapPartitionsRDD[178] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |   SQLExecutionRDD[177] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |   MapPartitionsRDD[176] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |   MapPartitionsRDD[174] at checkpoint at NativeMethodAccessorImpl.java:0 []
 |   ReliableCheckpointRDD[175] at checkpoint at NativeMethodAccessorImpl.java:0 []

Sample of final DataFrame values after iterative updates:
+----------+-------------+------------------+------------------+-----------+
|vehicle_id|vehicle_model|engine_temperature|battery_efficiency|distance_km|
+----------+-------------+------------------+------------------+-----------+
|VH_0000   |Tesla Model 3|61.196            |0.9815            |95.416     |
|VH_0000   |Tesla Model 3|60.4764     

## §17 — Lazy Evaluation

> 📌 Assignment Mapping — Part 4, Q1: Lazy Evaluation and DAG Construction

Every PySpark transformation you have applied in this notebook — `select`, `filter`, `withColumn`, `groupBy`, `agg`, `orderBy` — shares one property that is easy to miss until you watch a job in the Spark UI: **nothing runs when you write the transformation**. Spark records what you *want* to compute and waits.

That deferral is **lazy evaluation**. Transformations such as `map`, `filter`, and `groupBy` append nodes to a logical plan; they do not touch cluster CPU, memory, or disk. Only an **action** — `count`, `collect`, `show`, `write` — tells the driver to compile the accumulated plan and launch tasks.

The performance benefit is not subtle. Because Spark sees the *entire* plan before executing anything, the Catalyst optimiser can fuse adjacent narrow transforms, push filters closer to the source, and eliminate dead columns — optimisations that are impossible if each line of notebook code ran eagerly the moment it was written. In §9–11 you chained `select → filter → withColumn` before the first `groupBy`; Spark pipelined all three narrow steps into a single scan of the CSV rather than materialising three intermediate DataFrames.

You can feel the laziness directly: rebuild the §9–11 pipeline below and notice that calling `.groupBy(...).agg(...).orderBy(...)` produces no log output and no progress bar. The plan exists; execution waits for an action.


In [12]:
from pyspark.sql import functions as F

# ── Rebuild the §9–11 pipeline (transformations only — no action yet) ─────────
sdf_lazy = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/telemetry_raw.csv")
)

df_lazy_selected = sdf_lazy.select("vehicle_model", "engine_temperature")
df_lazy_filtered = df_lazy_selected.filter(
    (F.col("engine_temperature") >= 60.0) &
    (F.col("engine_temperature") <= 120.0)
)
df_lazy_with_f = df_lazy_filtered.withColumn(
    "engine_temp_f",
    F.round(F.col("engine_temperature") * 9 / 5 + 32, 2)
)
df_lazy_avg = (
    df_lazy_with_f
    .groupBy("vehicle_model")
    .agg(
        F.round(F.avg("engine_temperature"), 2).alias("avg_engine_temp_c"),
        F.count("*").alias("record_count"),
    )
)
df_lazy_result = df_lazy_avg.orderBy(F.col("avg_engine_temp_c").desc())

print("Transformations registered — no Spark job has run yet.")
print(f"Logical plan nodes exist; isStreaming = {df_lazy_result.isStreaming}")

# Action #1: materialise the plan — this is when Spark actually executes.
row_count = df_lazy_result.count()
print(f"\nAction triggered (count) — {row_count} vehicle models returned.")


Transformations registered — no Spark job has run yet.
Logical plan nodes exist; isStreaming = False



Action triggered (count) — 7 vehicle models returned.


## §18 — DAG Construction and Physical Execution Stages

Lazy evaluation does not mean Spark executes one transformation at a time in notebook order. When an action fires, the driver walks the logical plan and builds a **DAG** (Directed Acyclic Graph) of operations — the same graph you saw as RDD lineage in §14–15, now at the DataFrame/Catalyst level.

Use the §9–11 pipeline as the concrete example. Starting from `read CSV`, the DAG grows through three narrow transforms (`select`, `filter`, `withColumn`) that Spark can fuse into one map stage. Then `groupBy("vehicle_model").agg(...)` introduces the first **Wide Dependencies** shuffle: every partition that holds rows for "Tesla Model 3" must send those rows to the same reducer partition. Finally `orderBy(avg_engine_temp_c DESC)` introduces a second wide dependency — a global sort — and forces another shuffle.

**Wide Dependencies** are the reason Spark cuts the logical DAG into separate **physical execution Stages**. A narrow dependency can pipeline inside one task: partition *i* of the input feeds partition *i* of the output with no network traffic. A wide dependency cannot — reducer partition *j* may need rows from *every* mapper partition — so Spark must fully materialise the shuffle output (spill to disk, transfer over the network, read back on the receiving node) before the next stage's tasks can start. Each shuffle boundary is a hard cut: Stage 1 ends, shuffle completes, Stage 2 begins.

The §13 salting pipeline adds two more wide-dependency stage boundaries (salted `groupBy`, then de-salt `groupBy`) on top of the same principle — the DAG grows wider, but the stage-splitting rule does not change.

The code below prints the optimised physical plan for the §9–11 job and counts how many `Exchange` (shuffle) operators appear — each one marks a wide dependency and therefore a boundary between **physical execution Stages**.


In [13]:
# ── Physical plan + stage-boundary evidence on the real §9–11 pipeline ────────
spark.sparkContext.setJobGroup(
    "§17-18 DAG demo",
    "Show physical execution Stages for §9–11 avg-temperature job",
)

print("=== Optimised Physical Plan (§9–11 pipeline) ===")
plan_text = df_lazy_result._jdf.queryExecution().executedPlan().toString()
df_lazy_result.explain(mode="formatted")

# Each Exchange operator in the plan corresponds to one shuffle / stage boundary.
exchange_count = plan_text.count("Exchange")
stage_count  = exchange_count + 1   # N shuffles → N+1 physical execution Stages
print(f"\nExchange (shuffle) operators in plan : {exchange_count}")
print(f"Physical execution Stages (approx.)   : {stage_count}")
print("(Each Exchange marks a Wide Dependencies boundary — data must be")
print(" redistributed before the next physical execution Stage can start.)")


=== Optimised Physical Plan (§9–11 pipeline) ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [2]: [vehicle_model#1202, engine_temperature#1204]
Batched: false
Location: InMemoryFileIndex [file:/home/sabya/projects/pyspark-dev/data/telemetry_raw.csv]
PushedFilters: [IsNotNull(engine_temperature), GreaterThanOrEqual(engine_temperature,60.0), LessThanOrEqual(engine_temperature,120.0)]
ReadSchema: struct<vehicle_model:string,engine_temperature:double>

(2) Filter
Input [2]: [vehicle_model#1202, engine_temperature#1204]
Condition : ((isnotnull(engine_temperature#1204) AND (engine_temperature#1204 >= 60.0)) AND (engine_temperature#1204 <= 120.0))

(3) HashAggregate
Input [2]: [vehicle_model#1202, engine_temperature#1204]
Keys [1]: [vehicle_model#1202]
Functions [2]: [partial_avg(engine_temperat

## §19 — Data Locality and the I/O Bandwidth Tax

> 📌 Assignment Mapping — Part 4, Q2: Data Locality and the I/O Bandwidth Tax

Distributed systems have a hard speed limit that no amount of CPU can erase: moving bytes over a network is orders of magnitude slower than reading them from local disk, and slower still than reading them from RAM on the same socket.
 Spark's scheduling philosophy is therefore simple and ruthless: **Don't move data, move code**. Ship the task (the compiled bytecode for your `filter`, your `agg`, your UDF) to the node that already holds the input partition, rather than pulling the partition across the cluster to a central compute node.

### Data Locality

**Data Locality** is Spark's mechanism for honouring that philosophy. When the DAG scheduler assigns a task, it ranks candidate executor nodes by how close they are to the input split:

1. **PROCESS_LOCAL** — data is in memory on the same executor (best case).
2. **NODE_LOCAL** — data is on the same physical node, different JVM.
3. **RACK_LOCAL** — data is on the same rack, different node.
4. **ANY** — data must be fetched over the network from a remote rack.

For the telemetry CSV in §8, the first stage's `FileScan` tasks run where the file blocks live. Each task reads its local split, applies the fused narrow transforms (`select`, `filter`, `withColumn`), and emits shuffle data — all without a cross-cluster read of the raw CSV. Scheduling tasks on the node that already holds the data partition keeps network links free for the unavoidable wide-dependency shuffles in §9–11 and §13, where data *must* move because keys need to be co-located.

### Spark Lineage Recovery vs Hadoop Replication — the I/O Bandwidth Tax

Fault tolerance in Hadoop and Spark solves the same problem — survive node loss — but they pay for it at opposite ends of the pipeline.

Hadoop HDFS pays the **I/O bandwidth tax** **up front and repeatedly**: every block is written to three physical replicas on three separate nodes at ingest time. Every MapReduce stage reads those replicas from disk and writes new replicas for its output. You pay the network cost whether or not a node ever fails — three copies of 43 TB of telemetry is 129 TB of replication traffic before anyone runs a single query.

Spark inverts that trade. It keeps one logical copy of each partition and records the **lineage** — the DAG from §18 — as the recovery plan. If a task fails, Spark recomputes only the lost partition by replaying the lineage on a surviving node. No standing replication tax. The cost arrives only on failure, and it scales with lineage depth (which is why §16 introduces checkpointing to cap that depth).

The contrast in bandwidth terms:

| Strategy | When network I/O is spent | Cost on happy path |
|----------|--------------------------|--------------------|
| HDFS replication | At every write; every MapReduce shuffle | High — constant **I/O bandwidth tax** |
| Spark lineage | Only when a partition is lost and must be recomputed | Low — pay only on failure |

Spark still pays the **I/O bandwidth tax** during wide-dependency shuffles — that is unavoidable when keys must be co-located — but it avoids the standing multi-copy replication overhead that Hadoop carries on every byte, every day.


## §16b — Liability of Lineage

> 📌 Assignment Mapping — Part 4, Q3: Liability of Lineage

§14–15 established that Spark's lineage graph is the engine of **fault tolerance** **without heavy data replication**. §16 showed checkpointing as the fix for deep lineage in iterative jobs. This section names the problem checkpointing solves: the **Liability of Lineage**.

### The Liability in an Iterative Loop

Return to the simulated feature-engineering loop in §16: 40 iterations, each adding two `withColumn` transforms on top of the previous DataFrame. Every iteration appends two new nodes to the lineage graph. After 40 iterations the DAG is ~80 transforms deep; after 200 it would be ~400 deep. The lineage that makes Spark cheap to store becomes expensive to *use* in two specific ways:

1. **Recovery time grows without bound.** If an executor dies on iteration 195 of   200, Spark must replay all 195 prior transformation steps to reconstruct the   lost partition — even though the intermediate results were deterministic and   could have been snapshotted long ago. The longer the chain, the slower every   failure recovery becomes.
2. **The lineage graph itself can crash the driver.** Spark's DAG scheduler walks   the lineage recursively when planning and optimising jobs. An sufficiently deep   chain overflows the JVM call stack, producing **StackOverflow errors** — the   exact failure mode §16 demonstrated at iteration 40 with checkpointing every   10 steps.

That compounding cost — storage is cheap, but replay and scheduler recursion are not — is the **Liability of Lineage**: the longer the chain, the more you owe.
### How Checkpointing Breaks the Family Tree

`df.checkpoint()` writes the current DataFrame to reliable storage (`CHECKPOINT_DIR`, set in the bootstrap cell) and **severs the lineage**. The checkpointed DataFrame's parent is the checkpoint file on disk, not the chain of 80 `withColumn` nodes that produced it. Spark forgets everything before the checkpoint.

Called every 10 iterations in §16, checkpointing bounds lineage depth at ~20 nodes regardless of total iteration count. Recovery after a failure replays at most 10 iterations back to the last checkpoint — not back to the CSV read. This **stabilizes recovery time** and prevents the scheduler from walking an unbounded graph.

### Checkpointing vs Caching — What Each Actually Does

§16 used `.cache()` on the source DataFrame *and* `.checkpoint()` inside the loop. They look similar — both keep data around for reuse — but they solve different problems:

| | `.cache()` / `.persist()` | `.checkpoint()` |
|---|---|---|
| **Purpose** | Speed up *repeated reads* of the same DataFrame within a session | Truncate lineage for *fault tolerance* in long chains |
| **Storage** | Memory (spill to disk if memory pressure) | Reliable filesystem (`CHECKPOINT_DIR`) |
| **Lineage** | **Not truncated** — the full DAG remains intact | **Truncated** — parent becomes the checkpoint file |
| **On failure** | If the cached copy is lost (executor evicted, node died), Spark must **replay the full lineage** from the original source to rebuild it | Spark reads the checkpoint file directly — **no lineage replay** required |
| **Cost** | Cheap for short pipelines; memory pressure on large DataFrames | Disk I/O on every checkpoint call; pays off when lineage depth would otherwise grow unbounded |

**Cache** answers: "I am going to read this result again soon — keep it hot."
 **Checkpoint** answers: "This chain is getting too long — cut the history here so recovery does not depend on replaying everything that came before."

In the §16 loop, caching the CSV baseline avoids re-reading the file on every iteration (performance). Checkpointing every 10 iterations caps the **Liability of Lineage** (resilience). You need both, for different reasons — conflating them is one of the most common Spark mistakes in production iterative pipelines.
